# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pankaj1281/flyrank_ml_internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I use a Random Forest classifier because the task is to identify content likely to decline. It can capture nonlinear relationships between search performance, content characteristics, and freshness without requiring a simple linear relationship. The model is used for ranking and decision support rather than causal conclusions.

In [18]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

print("Model: Random Forest Classifier")
print("Number of trees:", model.n_estimators)

Model: Random Forest Classifier
Number of trees: 200


## 2. Split design

I use a client-grouped split so that content from the same client does not appear in both training and test sets. The split produced 25 training clients and 7 test clients, with 23,837 training rows and 6,163 test rows. No clients are shared between the two sets, making this a more honest test of generalization to unseen clients.

In [19]:
from sklearn.model_selection import GroupShuffleSplit

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "Train clients:",
    train_df["client_id"].nunique()
)

print(
    "Test clients:",
    test_df["client_id"].nunique()
)

print(
    "Clients shared between train/test:",
    len(
        set(train_df["client_id"]) &
        set(test_df["client_id"])
    )
)

Training rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Clients shared between train/test: 0


## 3. Train + compare vs my baseline

The Random Forest was evaluated on the same client-grouped test set as the baseline. The model achieved higher accuracy, precision, recall, and F1 than the majority-class baseline, indicating stronger predictive performance on the held-out clients.

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

In [21]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

In [22]:
X_train = train_df[numeric_features + categorical_features]
y_train = train_df["is_declining_label"]

X_test = test_df[numeric_features + categorical_features]
y_test = test_df["is_declining_label"]

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

model_pipeline.fit(X_train, y_train)

pred = model_pipeline.predict(X_test)

In [23]:
model_results = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall": recall_score(y_test, pred, zero_division=0),
    "f1": f1_score(y_test, pred, zero_division=0)
}

print(model_results)

{'accuracy': 0.8140515982476066, 'precision': 0.7882014388489209, 'recall': 0.8697999364877739, 'f1': 0.8269927536231884}


In [24]:
majority_class = y_train.mode()[0]

baseline_pred = [majority_class] * len(y_test)

baseline_results = {
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(
        y_test,
        baseline_pred,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        baseline_pred,
        zero_division=0
    ),
    "f1": f1_score(
        y_test,
        baseline_pred,
        zero_division=0
    )
}

print("Baseline:", baseline_results)

Baseline: {'accuracy': 0.5109524582184002, 'precision': 0.5109524582184002, 'recall': 1.0, 'f1': 0.676331615120275}


In [25]:
comparison = pd.DataFrame(
    [baseline_results, model_results],
    index=["Baseline", "Random Forest"]
)

comparison

,accuracy,precision,recall,f1
Baseline,0.510952,0.510952,1.0000,0.676332
Random Forest,0.814052,0.788201,0.8698,0.826993


## 4. Errors and interpretation

I check the incorrect predictions to understand where the model makes mistakes and whether the errors are concentrated in particular content types or performance patterns.

In [26]:
error_df = test_df[
    y_test.values != pred
].copy()

print("Incorrect predictions:", len(error_df))

print(
    error_df[
        [
            "content_id",
            "client_id",
            "content_type",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(10)
)

Incorrect predictions: 1146
               content_id          client_id     content_type   ctr  \
13   content_a5a2fbc76336  client_8527a891e2  keyword article  0.00   
26   content_72c5c2d73e5a  client_4e07408562  keyword article  0.12   
36   content_bce275871a25  client_f369cb89fc  keyword article  1.35   
129  content_b4170c25efd2  client_4e07408562  keyword article  0.05   
135  content_670746e86425  client_4e07408562  keyword article  0.32   
148  content_033581b09704  client_4e07408562  keyword article  0.07   
165  content_eaea09d6891e  client_4e07408562  keyword article  0.00   
204  content_976d5deeab73  client_4e07408562  keyword article  0.10   
252  content_aba4b4460e47  client_e629fa6598  keyword article  0.00   
277  content_714b0fd9ee80  client_e629fa6598  keyword article  0.18   

     avg_position  days_since_last_update  
13           39.8                     103  
26           30.0                      13  
36            5.4                      20  
129          2

In [27]:
print(
    error_df["content_type"].value_counts()
)

content_type
keyword article    1146
Name: count, dtype: int64


### Interpretation

The model made 1,146 incorrect predictions, all observed in keyword articles. The errors occur across different CTR, search-position, and freshness values, suggesting that the model uses multiple signals rather than relying on one simple rule. This is an observed error pattern, not evidence that content type causes prediction errors.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.